In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from metric import precision_at_recall

In [ ]:
train = pd.read_csv('train_features.csv', parse_dates=['window_start_ts'])
test = pd.read_csv('test_features.csv')
feature_cols = [col for col in test.columns if col != 'cookie_id']

In [ ]:
split_date = train['window_start_ts'].quantile(0.8)
train_part = train[train['window_start_ts'] < split_date].sort_values('window_start_ts')
valid_part = train[train['window_start_ts'] >= split_date]

X_train, y_train = train_part[feature_cols], train_part['target']
X_val, y_val = valid_part[feature_cols], valid_part['target']
print(f'граница: {split_date:%d.%m}, train: {len(train_part)}, valid: {len(valid_part)}')

In [ ]:
# baseline
# 14 признаков
# P@R0.7 = 0.463

# model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
# model.fit(X_train, y_train)
# pred = model.predict_proba(X_val)[:, 1]
#
# print(f"P@R0.7 = {precision_at_recall(y_val, pred):.3f}")

In [ ]:
# 14 признаков
# P@R0.7 = 0.464

# model = LGBMClassifier(n_estimators=1500, learning_rate=0.05, num_leaves=15, min_child_samples=30,
#                        subsample=0.8, subsample_freq=1, colsample_bytree=0.7, random_state=42, verbose=-1)
# model.fit(X_train, y_train)
# pred_lgbm = model.predict_proba(X_val)[:, 1]
# print(f"P@R0.7 = {precision_at_recall(y_val, pred_lgbm):.3f}")

In [ ]:
# 14 признаков
# P@R0.7 = 0.507

# model = XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=4, subsample=0.8,
#                       colsample_bytree=0.7, random_state=42)
# model.fit(X_train, y_train)
# pred_xgboost = model.predict_proba(X_val)[:, 1]
# print(f"P@R0.7 = {precision_at_recall(y_val, pred_xgboost):.3f}")

Вывод: разные модели и перебор гиперпараметров не дают особых изменений, поэтому нужно работать с признаками.
Всего получилось 21 признак, добавил признаки курсора, повторы объявлений и смены города

In [ ]:
# xgboost, 22 признак
# P@R0.7 = 0.738

# model = XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=4, subsample=0.8,
#                       colsample_bytree=0.7, random_state=42)
# model.fit(X_train, y_train)
# pred_xgboost = model.predict_proba(X_val)[:, 1]
# print(f"P@R0.7 = {precision_at_recall(y_val, pred_xgboost):.3f}")

In [ ]:
# catboost, 22 признак
# P@R0.7 = 0.692

# model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
# model.fit(train_part[feature_cols], train_part['target'])
# pred = model.predict_proba(valid_part[feature_cols])[:, 1]
#
# print(f"P@R0.7 = {precision_at_recall(y_val, pred):.3f}")

In [ ]:
# lightgbm, 22 признак
# P@R0.7 = 0.714

# model = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=15, min_child_samples=30,
#                        subsample=0.8, subsample_freq=1, colsample_bytree=0.7, random_state=42, verbose=-1)
# model.fit(X_train, y_train)
# pred_lgbm = model.predict_proba(X_val)[:, 1]
# print(f"P@R0.7 = {precision_at_recall(y_val, pred_lgbm):.3f}")

# Прогоняем через сетку гиперпараметров

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer

param_grid = {
    'max_depth': [3, 4, 6],
    'learning_rate': [0.03, 0.1],
    'n_estimators': [300, 700],
    'min_child_weight': [1, 5],
}
search = GridSearchCV(
    XGBClassifier(subsample=0.8, colsample_bytree=0.7, random_state=42),
    param_grid,
    scoring={'pr_auc': 'average_precision',
             'p_at_r': make_scorer(precision_at_recall, response_method='predict_proba')},
    refit='pr_auc',
    cv=TimeSeriesSplit(n_splits=3),
    n_jobs=1,
)
search.fit(X_train, y_train)
print('лучшие параметры:', search.best_params_)

results = pd.DataFrame(search.cv_results_)
cols = ['param_max_depth', 'param_learning_rate', 'param_n_estimators', 'param_min_child_weight',
        'mean_test_pr_auc', 'mean_test_p_at_r']
results.sort_values('mean_test_pr_auc', ascending=False)[cols].head(10).round(3)

In [ ]:
pred_xgboost_tuned = search.best_estimator_.predict_proba(X_val)[:, 1]
print(f"P@R0.7 = {precision_at_recall(y_val, pred_xgboost_tuned):.3f}")

In [ ]:
# ансамбль из catboost и xgboost дал лучший результат
# P@R0.7 = 0.794

catboost = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
catboost.fit(X_train, y_train)
pred_catboost = catboost.predict_proba(X_val)[:, 1]

xgboost = XGBClassifier(**search.best_params_, subsample=0.8, colsample_bytree=0.7, random_state=42)
xgboost.fit(X_train, y_train)
pred_xgboost = xgboost.predict_proba(X_val)[:, 1]

pred_ensemble = (pred_catboost + pred_xgboost) / 2
print(f'P@R0.7 = {precision_at_recall(y_val, pred_ensemble):.3f}')

In [ ]:
# сабмит обучаем на всём train и выдаем разультат на тесте

catboost_final = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
catboost_final.fit(train[feature_cols], train['target'])

xgboost_final = XGBClassifier(**search.best_params_, subsample=0.8, colsample_bytree=0.7, random_state=42)
xgboost_final.fit(train[feature_cols], train['target'])

score = (catboost_final.predict_proba(test[feature_cols])[:, 1]
         + xgboost_final.predict_proba(test[feature_cols])[:, 1]) / 2

submission = pd.DataFrame({'cookie_id': test['cookie_id'], 'score': score})
submission.to_csv('submission.csv', index=False)

Вывод: добавление фичей дало огромный прирост

# Важность признаков

In [ ]:
import matplotlib.pyplot as plt

importance_catboost = pd.Series(catboost.get_feature_importance(), index=feature_cols)
importance_xgboost = pd.Series(xgboost.feature_importances_, index=feature_cols)

importance = pd.DataFrame({
    'catboost': importance_catboost / importance_catboost.sum(),
    'xgboost': importance_xgboost / importance_xgboost.sum(),
})
importance['ensemble'] = importance.mean(axis=1)
importance = importance.sort_values('ensemble')

importance['ensemble'].plot(kind='barh', figsize=(7, 7), title='Важность признаков ансамбля')
plt.tight_layout()
plt.show()

importance.sort_values('ensemble', ascending=False).round(3)